In [1]:
import pandas as pd                      
import numpy as np                                   
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df=pd.read_csv("../clean_csv/processed_backorder_data.csv")
df

C:\Users\pc\AppData\Local\Temp\ipykernel_21868\3399650165.py:1: DtypeWarning: Columns (0: sku) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv("data/processed_backorder_data.csv")


,sku,national_inv,lead_time,in_transit_qty,forecast_3_month,forecast_6_month,forecast_9_month,sales_1_month,sales_3_month,sales_6_month,sales_9_month,min_bank,potential_issue,pieces_past_due,perf_6_month_avg,perf_12_month_avg,local_bo_qty,oe_constraint,went_on_backorder
0,1026827,0.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No,0.0,0.85,0.83,0.0,No,No
1,1043384,2.0,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No,0.0,0.99,0.99,0.0,No,No
2,1043696,2.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No,0.0,0.85,0.83,0.0,No,No
3,1043852,7.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,No,0.0,0.10,0.13,0.0,No,No
4,1044048,8.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,2.0,No,0.0,0.85,0.83,0.0,No,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1687856,1373987,-1.0,8.0,0.0,5.0,7.0,9.0,1.0,3.0,3.0,8.0,0.0,No,0.0,0.85,0.83,1.0,No,No
1687857,1524346,-1.0,9.0,0.0,7.0,9.0,11.0,0.0,8.0,11.0,12.0,0.0,No,0.0,0.86,0.84,1.0,No,Yes
1687858,1439563,62.0,9.0,16.0,39.0,87.0,126.0,35.0,63.0,153.0,205.0,12.0,No,0.0,0.86,0.84,6.0,No,No
1687859,1502009,19.0,4.0,0.0,0.0,0.0,0.0,2.0,7.0,12.0,20.0,1.0,No,0.0,0.73,0.78,1.0,No,No


5. How do professional companies avoid this problem?

Instead of returning only Yes or No, they return a risk score.

| Product | Backorder Probability | Recommendation                       |
| ------- | --------------------: | ------------------------------------ |
| A       |                    2% | No action needed                     |
| B       |                   35% | Monitor inventory                    |
| C       |                   78% | Review inventory and supplier status |
| D       |                   97% | Immediate action recommended         |


This gives planners much more useful information than a simple binary prediction.

In [3]:
# Feature 1: Total Available Supply
df['available_supply']=(df['national_inv']+df['in_transit_qty'])
#Meaning
#Current inventory + inventory currently coming.

In [4]:
# Feature 2 :Supply Gap
#Compare available supply with expected demand.
df['supply_gap_3m']=(df['available_supply']-df['forecast_3_month'])

""" 
Interpretation:
Negative = potential shortage.

Positive = supply exceeds forecast demand
"""

' \nInterpretation:\nNegative = potential shortage.\n\nPositive = supply exceeds forecast demand\n'

In [5]:
# Feature 3:Supply/Demand Ratio
df['supply_demand_ratio_3m']=(df['available_supply']/(df['forecast_3_month']+1))
"""  
Example : 
available_supply:100
forcast :200
ratio :0.5 ->Available supply represents approximately 50% of forecast demand.

ratio < 1 : Supply is lower than forecast demand.

ratio ≈ 1 : Supply approximately matches forecast demand.

ratio > 1 : Supply exceeds forecast demand.
"""

'  \nExample : \navailable_supply:100\nforcast :200\nratio :0.5 ->Available supply represents approximately 50% of forecast demand.\n\nratio < 1 : Supply is lower than forecast demand.\n\nratio ≈ 1 : Supply approximately matches forecast demand.\n\nratio > 1 : Supply exceeds forecast demand.\n'

In [6]:
# Feature 4:Inventory vs Minimum Bank
df['inventory_min_gap']=(df['national_inv']-df['min_bank'])

""" 
Example:

inventory = 20
min_bank = 50

gap = -30

Meaning:

Inventory is 30 units below the company's minimum desired level.
"""

df['below_min_bank']=(df['national_inv']<df['min_bank']).astype(int)
"""
So:

0 = inventory is not below minimum
1 = inventory is below minimum
"""

'\nSo:\n\n0 = inventory is not below minimum\n1 = inventory is below minimum\n'

In [7]:
# Feature 5:Forecast Growth / Long-Term Demand
df['forcast_growth_3_to_6']=(df['forecast_6_month']-df['forecast_3_month'])
df['forcast_growth_6_to_9']=(df['forecast_9_month']-df['forecast_6_month'])

df["forecast_monthly_3m"] = (df["forecast_3_month"] / 3)
df["forecast_monthly_6m"] = (df["forecast_6_month"] / 6)

df["forecast_monthly_9m"] = (df["forecast_9_month"] / 9)

In [ ]:
# feature 6:Historical Sales Velocity
df["sales_monthly_3m"] = (df["sales_3_month"] / 3)

df["sales_monthly_6m"] = (df["sales_6_month"] / 6)

df["sales_monthly_9m"] = (df["sales_9_month"] / 9)

In [14]:
df['forcast_vs_sales']=(df['forecast_monthly_3m']/(df['sales_monthly_3m']+1))
"""  
Example:

Historical monthly sales = 50
Forecast monthly demand = 100

ratio = 2

Interpretation:

The expected demand is approximately twice the recent historical sales rate.

This could identify products where demand is expected to increase.
"""

'  \nExample:\n\nHistorical monthly sales = 50\nForecast monthly demand = 100\n\nratio = 2\n\nInterpretation:\n\nThe expected demand is approximately twice the recent historical sales rate.\n\nThis could identify products where demand is expected to increase.\n'

In [17]:
# Feature 10:Inventory Coverage
#How many months of expected demand can our current inventory cover
df["inventory_coverage_months"] = (df["national_inv"] /(df["forecast_monthly_3m"] + 1))
"""  
Example:

Inventory = 100
Monthly forecast = 50

Coverage = 2 months

Meaning:

Current inventory could theoretically cover approximately 2 months of forecast demand, assuming demand and supply conditions remain comparable.

This is much more intuitive for a supply-chain manager.
"""

'  \nExample:\n\nInventory = 100\nMonthly forecast = 50\n\nCoverage = 2 months\n\nMeaning:\n\nCurrent inventory could theoretically cover approximately 2 months of forecast demand, assuming demand and supply conditions remain comparable.\n\nThis is much more intuitive for a supply-chain manager.\n'

In [11]:
# Feature 11 :Inventory Coverage Including Incoming Stock
#Are we likely to run out before replenishment arrives?
df["total_supply_coverage_months"] = (df["available_supply"] /(df["forecast_monthly_3m"] + 1))

In [18]:
df

,sku,national_inv,lead_time,in_transit_qty,forecast_3_month,forecast_6_month,forecast_9_month,sales_1_month,sales_3_month,sales_6_month,...,forcast_growth_6_to_9,forecast_monthly_3m,forecast_monthly_6m,forecast_monthly_9m,sales_monthly_3m,sales_monthly_6m,sales_monthly_9m,forcast_vs_sales,inventory_coverage_months,total_supply_coverage_months
0,1026827,0.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,1043384,2.0,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.000000,2.000000
2,1043696,2.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.000000,2.000000
3,1043852,7.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,7.000000,7.000000
4,1044048,8.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.444444,0.000000,8.000000,8.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1687856,1373987,-1.0,8.0,0.0,5.0,7.0,9.0,1.0,3.0,3.0,...,2.0,1.666667,1.166667,1.000000,1.000000,0.500000,0.888889,0.833333,-0.375000,-0.375000
1687857,1524346,-1.0,9.0,0.0,7.0,9.0,11.0,0.0,8.0,11.0,...,2.0,2.333333,1.500000,1.222222,2.666667,1.833333,1.333333,0.636364,-0.300000,-0.300000
1687858,1439563,62.0,9.0,16.0,39.0,87.0,126.0,35.0,63.0,153.0,...,39.0,13.000000,14.500000,14.000000,21.000000,25.500000,22.777778,0.590909,4.428571,5.571429
1687859,1502009,19.0,4.0,0.0,0.0,0.0,0.0,2.0,7.0,12.0,...,0.0,0.000000,0.000000,0.000000,2.333333,2.000000,2.222222,0.000000,19.000000,19.000000


In [19]:
df.isnull().sum()

sku                             0
national_inv                    0
lead_time                       0
in_transit_qty                  0
forecast_3_month                0
forecast_6_month                0
forecast_9_month                0
sales_1_month                   0
sales_3_month                   0
sales_6_month                   0
sales_9_month                   0
min_bank                        0
potential_issue                 0
pieces_past_due                 0
perf_6_month_avg                0
perf_12_month_avg               0
local_bo_qty                    0
oe_constraint                   0
went_on_backorder               0
available_supply                0
supply_gap_3m                   0
supply_demand_ratio_3m          0
inventory_min_gap               0
below_min_bank                  0
forcast_growth_3_to_6           0
forcast_growth_6_to_9           0
forecast_monthly_3m             0
forecast_monthly_6m             0
forecast_monthly_9m             0
sales_monthly_

In [ ]:
df.to_csv("../clean_csv/backorder_with_new_feature.csv", index=False)

# Author
### <a href="https://www.linkedin.com/in/salma-belaicha-04a646336/" target="_blank">Salma Belaicha</a>